# PokerMind — live driver

Calls the real agents and Judge **now**. Each section spends tokens; the `*_tokens` cell after it shows how many. Change `SEED` to play a fresh game (re-running the same `SEED` replays the three-tournament section from cache).

## Setup

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd()
ROOT = ROOT.parents[1] if ROOT.name == "notebooks" else ROOT
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

In [2]:
from task_3.pokermind.utils import prepare_notebook, report
from task_3.pokermind.observability import initialize_observability
from task_3.pokermind.experiment import run_observation_hand, run_one_tournament, run_final_experiment
RESULTS, _, _ = prepare_notebook(ROOT)
telemetry = initialize_observability()

In [3]:
SEED = 20260722     # change this number to play a different game
BUTTON = 0
("Ready — OPENROUTER_API_KEY is set." if os.environ.get("OPENROUTER_API_KEY")
 else "Set OPENROUTER_API_KEY before running the sections below.")

'Ready — OPENROUTER_API_KEY is set.'

## 1 · One hand

Plays a single hand and shows every agent's action and reason.

In [4]:
_, hand = run_observation_hand(f"live_hand_{SEED}", SEED, BUTTON, telemetry)
report.hand_actions(hand["hands"][0])

,Player,Street,Action,Amount,Reason
0,Conservative,preflop,CALL,50,As-Ts on the button is a strong hand worth seeing a flop. Calling is affordable and keeps me in position with a suited ace.
1,Aggressive,preflop,CALL,25,Getting 5:1 pot odds to complete the blind with a weak hand; limpers showed no strength preflop.
2,Adaptive,preflop,CHECK,0,KTo is decent but not premium; checking from the big blind gives a free flop with position to act after both limpers. No need to inflate pot with marginal hand early in tournament.
3,Aggressive,flop,CHECK,0,"Missed flop completely (5h8c on 4d Ah 2d). No pair, no draw. Checking to see a free turn card with three players in the hand."
4,Adaptive,flop,CHECK,0,"King-high with no pair on an A-high board. Both opponents checked, but betting into two limpers with KTo on this flop is thin. Taking the free card to see the turn and potentially improve (backdoor flush/straight draws)."
5,Conservative,flop,RAISE,100,"Top pair top kicker on A42 flop. Betting for value and to charge flush draws. Pot is 150, a 100 raise is a reasonable size."
6,Aggressive,flop,FOLD,0,Weak 5-high hand with no draws facing a raise from Conservative into two players. Not worth continuing.
7,Adaptive,flop,FOLD,0,K-high with no pair or strong draw on A-high flop; Conservative raised into two opponents showing strength. Calling 100 with weak hand is -EV.


In [5]:
report.hand_tokens(hand["hands"][0])

Decisions             8
Input tokens     33,079
Output tokens     5,031
Total tokens     38,110
Name: one hand, dtype: object

## 2 · One tournament

Plays a full tournament (and its Judges) to a single winner.

In [6]:
_, tourney, _ = run_one_tournament(f"live_{SEED}", SEED, BUTTON, RESULTS, telemetry)
report.tournament_result(tourney)

Winner          Aggressive
Hands played             9
Name: result, dtype: object

In [7]:
report.player_metrics(tourney)

,Place,Hands won,Fold,Call,Raise,All-ins,End stack
Aggressive,1,6,11%,16%,37%,2,1500
Conservative,2,2,33%,28%,6%,0,0
Adaptive,3,1,33%,22%,22%,1,0


In [8]:
report.player_tokens(tourney)

Model calls           48
Input tokens     517,964
Output tokens    162,913
Total tokens     680,877
Name: one tournament, dtype: object

### Decision by decision — first & last 2 hands

How the agents actually played, in reading order.

In [9]:
report.show_hands(tourney, first=2, last=2)

── Hand 1  ·  blinds 25/50  ·  board 4d Ah 2d
      Conservative  As Ts
      Aggressive    5h 8c
      Adaptive      Kc Td
   preflop  Conservative  CALL 50
            ↳ On the button with As-Ts suited, a strong starting hand. Calling 50 to see the flop is affordable and keeps the pot manageable while in position.
   preflop  Aggressive    CALL 25
            ↳ Getting 5:1 pot odds to call 25 more with weak hand; cheap flop in multi-way pot.
   preflop  Adaptive      RAISE 150
            ↳ KTo is strong enough to raise limpers from big blind. Raise to 150 (pot-sized) to isolate or take it down now. Both opponents have only called so far, suggesting passive play.
   preflop  Conservative  CALL 150
            ↳ AT suited is strong enough to call the big blind's raise, getting 2:1 pot odds with position on the button.
   preflop  Aggressive    FOLD
            ↳ Weak 5-8 offsuit hand facing a raise and call, out of position, and calling would commit too much of my remaining stack.
   

## 3 · LLM judge — one tournament

An independent Judge (different model) scores each agent 1–10 on the run above, with a written rationale for every score.

In [10]:
report.judge_scores(tourney)

,Decision,Strategy,Adaptation,Reason,Tool,Overall,Calls
Conservative,8,9,7,9,—,8,1
Aggressive,9,10,9,9,—,9,1
Adaptive,8,9,9,9,7,8,1


In [11]:
report.judge_reasoning(tourney)

Score  \
Agent        Aspect                   
Conservative Decision             8   
             Strategy             9   
             Adaptation           7   
             Reason               9   
             Tool                 —   
             Main improvement     8   
Aggressive   Decision             9   
             Strategy            10   
             Adaptation           9   
             Reason               9   
             Tool                 —   
             Main improvement     9   
Adaptive     Decision             8   
             Strategy             9   
             Adaptation           9   
             Reason               9   
             Tool                 7   
             Main improvement     8   

                                                                                                                                                                                                                                                                                                                                                                                                                                                            Reasoning  
Agent        Aspect                                                                                                                                                                                                                                                                                                                                                                                                                                                    
Conservative Decision                                                          The player demonstrated disciplined decision-making, particularly in Hand 1 where they extracted value with top pair and in Hand 7 where they correctly folded Ace-high on a coordinated board after facing turn aggression. The player avoided over-committing with marginal holdings like J5o and Q3o, showing a good grasp of pot odds and hand strength relative to board texture.  
             Strategy                                                                                                               The player adhered closely to the 'Conservative' strategy. They consistently called small blind differences to see flops (Hand 6, Hand 8) but were quick to fold when facing significant pressure without a made hand. The check-fold lines with marginal holdings perfectly match the patient, lower-variance profile described.  
             Adaptation                                                                                  The player adjusted well to short-stack heads-up play by seeing cheap flops but maintained their tight-aggressive standards for committing their stack. However, in Hand 7, checking back AQs preflop in the big blind might be overly passive even for a conservative strategy, missing an opportunity to put pressure on the opponent's wide button range.  
             Reason                                                                                                                                                           The rationales provided are logically sound and reference specific game states, such as pot odds (Hand 1, Hand 8), board coordination (Hand 6, Hand 7), and stack sizes. The reasoning for folding 9To in Hand 2 correctly identified the risk of a multi-way pot with a marginal hand.  
             Tool                                                                                                                                                                                                                                                                                                                                                                                      No tools were used or available in the provided decision logs.  
             Main improvement                            

In [12]:
report.judge_tokens(tourney)

Judge calls           3
Input tokens     16,595
Output tokens     1,968
Total tokens     18,563
Name: judge · one tournament, dtype: object

## 4 · Three tournaments

Runs the full experiment (three tournaments, rotating the dealer), then shows per-tournament stats and a summary across all three.

In [13]:
_, summary, _ = run_final_experiment(SEED, RESULTS, telemetry)
tours = report.load_tournaments(summary, RESULTS)
report.experiment_tournaments(summary)

,Winner,Hands,Player tokens,Judge calls,Judge tokens,Tool calls
Tournament 1,Aggressive,8,"805,863",3,"20,396",37
Tournament 2,Aggressive,7,"518,523",3,"15,314",23
Tournament 3,Aggressive,7,"370,783",3,"13,625",11


**Per-tournament** — one agent table per tournament.

In [14]:
report.tournament_stats(tours[0], 1)

,Place,Hands won,Fold,Call,Raise,All-ins,End stack
Aggressive,1,5,17%,33%,33%,1,1500
Conservative,2,2,29%,36%,7%,1,0
Adaptive,3,1,33%,20%,13%,0,0


In [15]:
report.tournament_stats(tours[1], 2)

,Place,Hands won,Fold,Call,Raise,All-ins,End stack
Aggressive,1,6,11%,11%,56%,0,1500
Conservative,2,0,20%,50%,0%,1,0
Adaptive,3,1,31%,23%,15%,1,0


In [16]:
report.tournament_stats(tours[2], 3)

,Place,Hands won,Fold,Call,Raise,All-ins,End stack
Aggressive,1,6,0%,20%,40%,0,1500
Conservative,2,2,27%,33%,0%,2,0
Adaptive,3,0,40%,20%,20%,1,0


**Summarized across the three tournaments.**

In [17]:
report.experiment_agents(summary)

,Tournament wins,Avg place,Hands won,Raise,Player tokens
Conservative,0,2.0,4,3%,"368,686"
Aggressive,3,1.0,17,42%,"319,187"
Adaptive,0,3.0,2,15%,"1,007,296"


In [18]:
report.experiment_tokens(summary)

Model calls            108
Tool calls              71
Input tokens     1,355,757
Output tokens      339,412
Total tokens     1,695,169
Name: three tournaments, dtype: object

## 5 · LLM judges — three tournaments

Judge scores per tournament (with judge calls), a summary, and the written notes.

**Per-tournament judge scores** — one table per tournament.

In [19]:
report.tournament_judges(tours[0], 1)

,Decision,Strategy,Adaptation,Reason,Tool,Overall,Calls
Conservative,9,10,8,9,—,9,1
Aggressive,8,9,9,10,—,9,1
Adaptive,8,9,9,9,7,8,1


In [20]:
report.tournament_judges(tours[1], 2)

,Decision,Strategy,Adaptation,Reason,Tool,Overall,Calls
Conservative,6,8,5,7,—,6,1
Aggressive,8,9,9,9,—,9,1
Adaptive,8,9,9,8,7,8,1


In [21]:
report.tournament_judges(tours[2], 3)

,Decision,Strategy,Adaptation,Reason,Tool,Overall,Calls
Conservative,8,10,7,9,—,8,1
Aggressive,9,10,9,10,—,9,1
Adaptive,9,10,10,9,8,9,1


**Summarized across the three tournaments.**

In [22]:
report.judge_averages(summary)

,Decision,Strategy,Adaptation,Reason,Tool,Overall
Conservative,7.7,9.3,6.7,8.3,—,7.7
Aggressive,8.3,9.3,9.0,9.7,—,9.0
Adaptive,8.3,9.3,9.3,8.7,7.3,8.3


In [23]:
report.experiment_judge_reasoning(summary, RESULTS)

Overall  \
Tournament Agent                   
1          Conservative        9   
           Aggressive          9   
           Adaptive            8   
2          Conservative        6   
           Aggressive          9   
           Adaptive            8   
3          Conservative        8   
           Aggressive          9   
           Adaptive            9   

                                                                                                                                                                                                                                                                                       Main improvement  
Tournament Agent                                                                                                                                                                                                                                                                                         
1          Conservative                                                            Consider incorporating more selective preflop aggression with premium hands like ATs or AQo to build pots when holding a range advantage, rather than relying primarily on passive calling and post-flop navigation.  
           Aggressive                                                                                                          Tighten the preflop calling range from the button against raises, as hands like K3o and Q8o can be easily dominated and lead to unnecessary chip loss in multi-way pots.  
           Adaptive                                 Improve resource management regarding one-time-use tools. Several decisions were wasted attempting to use the peek tool after it was already exhausted, which could have been better spent on deeper statistical analysis or alternative reasoning.  
2          Conservative  Improve short-stack tournament mechanics. Instead of calling preflop raises that leave an SPR (Stack-to-Pot Ratio) of less than 1, the player should look to either fold or shove to maximize fold equity and avoid being forced to fold for a tiny fraction of the pot later.  
           Aggressive                                                                                                           Continue to refine the bottom of the button-raising range to ensure that even 'aggressive' plays maintain a baseline of post-flop playability against multiple callers.  
           Adaptive                                                                                                                          Improve tracking of one-time tool usage to avoid wasted API calls, and tighten the small blind defending range against button raises when out of position.  
3          Conservative   Increase aggression with strong made hands when short-stacked. While the conservative approach preserves chips, checking top pairs on coordinated boards allows opponents to realize their equity for free, which is dangerous when the player's own stack is critically low.  
           Aggressive                                               The player's performance was highly effective for this field. To improve further, they could refine their bluff sizing on the flop to ensure they aren't over-committing against ranges that are likely to call down with top pair.  
           Adaptive                                                                                 Improve tracking of tool limits to avoid wasted calls to the peek tool, and ensure offsuit connectors like 98o are treated with more caution in early levels compared to their suited counterparts.

In [24]:
report.judge_experiment_tokens(summary)

Evaluations           9
Judge calls           9
Input tokens     43,893
Output tokens     5,442
Total tokens     49,335
Name: judges · three tournaments, dtype: object

## What we saw

We sat three computer players down at a poker table and told each of them to play in a different way:

- **Conservative** — careful. Mostly waits for good cards and only bets big when it's fairly sure.
- **Aggressive** — pushy. Bets and raises a lot to put pressure on the other two.
- **Adaptive** — the "reader." Tries to watch how the others play and adjust.

They played three full games (plus one extra single game at the top of this notebook). Here's what happened, in plain terms.

**The aggressive player won every single game.** These games are short and fast — each lasted only 7–9 hands, because the forced bets climb quickly and someone runs out of chips — and in that kind of quick game, being pushy paid off. The aggressive player raised on about 4 out of every 10 turns and almost never backed down (it folded only about 8% of the time). The careful player did the opposite: it raised only about 3% of the time and folded far more often — and it never won a game. The "reader" finished last in all three.

So the players really did behave the way we asked them to: the aggressive one was clearly the most aggressive, and the careful one was clearly the most careful.

**We also had a separate AI act as a neutral referee**, grading *how well* each player made its choices — not just whether it won, since poker has a lot of luck in it. The referee gave everyone fairly high marks out of 10: the aggressive player scored the best (around 9), and even the players who lost still scored around 8. In other words, the losers weren't playing badly — the aggressive style just tended to come out on top in these quick games.

**One thing worth pointing out:** the "reader" came last every time even though the referee thought its decisions were sensible. That's a good reminder that in poker, making good decisions doesn't guarantee you win — especially over just a handful of hands, where luck matters a lot.

**Roughly what it cost:** across the three games, the players and the referee together made a bit over a hundred requests to the AI and exchanged around 1.7 million tokens of text (a token is roughly a word).

**The honest bottom line:** this was only three games, so treat it as *"here's what we noticed,"* not *"aggressive always wins."* With so few hands, luck plays a big role. But it's a clear, encouraging sign of two things: the three players genuinely stick to their assigned styles, and the whole setup runs from the first card dealt to a final winner and an independent grade — start to finish, without falling over.

*(These observations describe this run. If you play a new game by changing the seed above, the numbers — and possibly the winner — will change.)*